# Optical communication: implementation walkthrough

This notebook explains the **final November screen-to-camera project**, from text to decoded message and then to experimental evidence. It can run from top to bottom without recordings, hardware or network access after installing the requirements.

The examples below are **synthetic software demonstrations**, not new physical results. Read `report.pdf` for the 21-video campaign. For new recordings, use `Record_Physical_Trials.ipynb` one cell at a time.

1. Load the implementation.
2. Trace framing, coding and transmission cost.
3. Check correction and interleaving with small examples.
4. Decode synthetic brightness through the actual receiver.
5. Inspect the fixed physical experiment design.
6. Locate separately supplied empirical evidence.

In [ ]:
from pathlib import Path
import sys
import json
import numpy as np
import pandas as pd
from IPython.display import display

candidates = [Path.cwd(), Path.cwd() / 'optical_link', Path.cwd().parent / 'optical_link']
PROJECT = next((p.resolve() for p in candidates if (p / 'code/optical_link/protocol.py').is_file()), None)
if PROJECT is None:
    raise RuntimeError('Open this notebook beside the project code/ directory.')
sys.path.insert(0, str(PROJECT / 'code'))
from optical_link.protocol import make_packet, encode_body, decode_body
from optical_link.coding_experiments import interleave, deinterleave
from optical_link.receiver import synthetic_trace
from optical_link.physical_campaign_receiver import decode_pilot
from optical_link.physical_campaign import load_sessions, wire_packet
print('Loaded the final implementation; no hardware is opened.')

## 1. Text becomes a framed, protected packet

UTF-8 bytes become bits. The body contains the message and its CRC, protected by the chosen code. A known preamble locates the packet; a header gives its format. Header and body checks serve different purposes: body correction cannot rescue a header that was never acquired.

At 200 ms per bit, adding redundancy increases airtime. Goodput also accounts for failures, so the perfect-delivery rates below are **upper bounds**, not measured results.

In [ ]:
rows = []
for codec in ['none', 'repeat3', 'repeat5', 'hamming74', 'rs4']:
    packet = make_packet('A7k2M9q4', codec, sequence=1)
    airtime = len(packet.bits) * 0.2
    rows.append({'code': codec, 'payload_bytes': len(packet.payload),
                 'body_bits': len(packet.body_bits), 'total_bits': len(packet.bits),
                 'airtime_s': airtime, 'perfect_delivery_bits_per_s': len(packet.payload)*8/airtime})
display(pd.DataFrame(rows))

## 2. Hamming repairs one error in each seven-bit word

Four data bits are protected by three parity bits. Their failed checks identify one changed position. Two or more errors in the same word exceed this guarantee. The receiver must still validate the recovered message.

In [ ]:
data = np.array([1, 0, 1, 1], dtype=np.uint8)
encoded = encode_body(data, 'hamming74')
received = encoded.copy()
received[4] ^= 1  # Flip position 5 (zero-based index 4).
recovered = decode_body(received, 'hamming74')
assert np.array_equal(recovered, data)
for label, bits in [('Data', data), ('Encoded', encoded), ('Received', received), ('Recovered', recovered)]:
    print(f'{label:10s}', ''.join(map(str, bits)))

## 3. Interleaving changes which groups contain the errors

This deliberately small example uses **depth three** to explain the mechanism; the physical study uses depth eight. Three adjacent wire errors are spread over three Hamming words, leaving one repairable error in each. Interleaving adds no transmitted bits. It can harm a byte-based code if spreading errors damages more bytes than that code can repair.

In [ ]:
data = np.tile(np.array([1, 0, 1, 1], dtype=np.uint8), 3)
encoded = encode_body(data, 'hamming74')
wire = interleave(encoded, depth=3)
received = wire.copy()
received[:3] ^= 1
restored = deinterleave(received, depth=3)
errors_per_word = (restored != encoded).reshape(-1, 7).sum(axis=1)
assert errors_per_word.tolist() == [1, 1, 1]
assert np.array_equal(decode_body(restored, 'hamming74'), data)
print('Errors per Hamming word after inverse ordering:', errors_per_word)
print('All three words recovered; total wire bits:', len(wire))

## 4. Exercise the actual receiver on synthetic brightness

The following input is generated in software. It checks the end-to-end decoding path at every code/depth combination in a main blockage session. The receiver is passed brightness, timestamps and known interleaving depth; expected text is used only afterwards to score its output.

A clean synthetic pass does not establish robustness to real glare, blockage or acquisition failure.

In [ ]:
sessions = load_sessions(PROJECT)
main_rows = next(group for group in sessions.values()
                 if group[0]['phase'] == 'main' and group[0]['condition'] == 'OCC_CLUSTER')
results = []
for row in main_rows:
    original, wire_packet_value = wire_packet(row)
    times, brightness = synthetic_trace(wire_packet_value, seed=8, drop_fraction=0, noise=0)
    result = decode_pilot(times, brightness, depth=int(row['interleave_depth']))
    exact = result.get('payload') == original.payload
    assert exact and result.get('crc_ok')
    results.append({'code': row['codec'], 'depth': row['interleave_depth'],
                    'exact_message': exact, 'CRC_valid': result['crc_ok']})
display(pd.DataFrame(results))

## 5. Inspect the physical design before interpreting results

The configuration contains 82 **planned transmissions** across 21 sessions. Reprocessing a recording with seven receivers does not create seven independent physical observations. Shared non-interleaved controls are counted once.

The report separates development, main comparisons and acquisition stress. Its main messages have 4, 8 and 12 bytes, each repeated in two rounds; one message per length limits separation of length and content effects.

In [ ]:
schedule = pd.DataFrame([row for group in sessions.values() for row in group])
assert len(schedule) == 82 and len(sessions) == 21
display(schedule.groupby(['phase', 'condition'], sort=False).agg(
    planned_packets=('packet_id', 'count'), planned_videos=('session_id', 'nunique')))
print('Schedule inspection is not evidence that a new recording run has completed.')

## 6. Review physical results only when the evidence is supplied

The clean repository excludes datasets. The optional cell below reads saved outcomes if they have been restored; otherwise it gives the correct next step and completes without error. Full reproduction, source verification and raw-video requirements are in `docs/REPRODUCING_RESULTS.md`.

Read the report alongside these outcomes: distinguish body errors from detection failures, raw BER from delivered messages, and physical observations from model-based simulation results.

In [ ]:
study = PROJECT / 'data/analysis/physical_campaign_2026-11-08'
packet_path = study / 'packets.json'
if not packet_path.is_file():
    print('Experimental evidence is intentionally separate from this source package.')
    print('See docs/REPRODUCING_RESULTS.md to restore a working copy for replay.')
else:
    packets = pd.DataFrame(json.loads(packet_path.read_text()))
    main = packets[packets['phase'] == 'main']
    display(main.groupby(['condition', 'codec', 'depth']).agg(
        packets=('success', 'count'), recovered=('success', 'sum')))
    print('Saved observations only; this cell does not re-decode videos or verify provenance.')

## 7. Next steps and implementation map

- **Read the findings:** `report.pdf`.
- **Understand every module:** `docs/ARCHITECTURE.md`.
- **Run the regression suite:** `PYTHONPATH=code .venv/bin/python -m unittest discover -s code/tests -v` from the project root.
- **Record a fresh campaign:** the numbered recording notebook and `docs/RECORDING_GUIDE.md`.
- **Replay measured evidence:** `docs/REPRODUCING_RESULTS.md`; data must be supplied separately.
- **Inspect earlier collaborator work:** `legacy/comparison_study/`, separate from the final implementation.

The main practical finding is to match the intervention to the failure: broader spatial measurement can address local glare; interleaving must suit a code's correction groups; failures before body decoding require acquisition improvements. More complex processing is useful only when the measured evidence supports it.